# Emotional Arc Generation — 1,627 Scripts

This notebook generates emotional arcs for all scripts in the final analysis corpus using `siebert/sentiment-roberta-large-english`.

**Two passes per script:**
1. **Main arc** — 20 windows (W = T/10, S = W/2), z-scored, normalized x-axis
2. **Secondary arc** — 1,500-token windows, 500-token step, Savitzky-Golay smoothed (for reversal detection later)

**Output:** Per-script JSONs in `outputs/arcs/` + consolidated `arcs_all.json`, saved to Google Drive.

**Runtime estimate:** ~4–6 hours on Colab T4 GPU. Checkpointing is built in — re-running skips already-completed scripts.

## 0. Setup

In [ ]:
!pip install -q transformers torch scipy tqdm

In [ ]:
# Clone the repo — this pulls all scripts, metadata, and the notebook itself
!git clone https://github.com/derinsavasan/thesis.git /content/thesis
print('Clone complete.')

In [ ]:
# Mount Drive — outputs are saved here so they survive session disconnects
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

REPO_ROOT     = Path('/content/thesis')
SCRIPTS_DIR   = REPO_ROOT / 'Movie-Script-Database/scripts/parsed/tagged'
METADATA_PATH = REPO_ROOT / 'Movie-Script-Database/scripts/metadata/analysis_corpus_final.json'

# Outputs go to Drive so they persist if the session disconnects
ARCS_DIR = Path('/content/drive/MyDrive/thesis-outputs/arcs')
ARCS_DIR.mkdir(parents=True, exist_ok=True)

script_files = list(SCRIPTS_DIR.glob('*_parsed.txt'))
print(f'Scripts found : {len(script_files)}')
print(f'Metadata found: {METADATA_PATH.exists()}')
print(f'Arcs output   : {ARCS_DIR}')

## 1. Load model

In [ ]:
import torch
from transformers import pipeline

MODEL_NAME = 'siebert/sentiment-roberta-large-english'

device = 0 if torch.cuda.is_available() else -1
print(f'Device: {"GPU" if device == 0 else "CPU — check Runtime > Change runtime type > T4 GPU"}')

pipe = pipeline(
    'text-classification',
    model=MODEL_NAME,
    device=device,
    truncation=True,
    max_length=512,
)

## 2. Helper functions

In [ ]:
import re
import numpy as np
from scipy.signal import savgol_filter


# Strip tag labels (S:, N:, C:, D:, E:, T:, M:) — keep content, include all line types
TAG_RE = re.compile(r'^[SNCDЕТM]:\s*', re.MULTILINE)

def extract_text(path: Path) -> str:
    return TAG_RE.sub('', path.read_text(encoding='utf-8', errors='replace'))


# Word-level tokenizer: one token per word OR punctuation mark
TOKEN_RE = re.compile(r'\w+|[^\w\s]', re.UNICODE)

def tokenize(text: str) -> list[str]:
    return TOKEN_RE.findall(text)


def _signed(result: dict) -> float:
    score = result['score']
    return score if result['label'].upper() == 'POSITIVE' else -score


SUBCHUNK_WORDS  = 400   # stays safely under 512 subword tokens
MIN_TOKENS_MAIN = 1000
SEC_WINDOW      = 1500
SEC_STEP        = 500
SG_WINDOW       = 5
SG_ORDER        = 2


def compute_arcs(tokens: list[str], batch_size: int = 32) -> tuple[list[dict], list[dict]]:
    """
    Compute both arcs for one script in a single pipe() call.

    All subchunks from all windows (main + secondary) are flattened into one
    list and scored together — saturates the GPU and eliminates the
    sequential-pipeline warning.
    """
    T = len(tokens)

    # Build main windows
    W = T // 10
    S = W // 2
    main_wins = []
    for i in range(20):
        start = i * S
        end   = min(start + W, T)
        chunk = tokens[start:end]
        if len(chunk) < MIN_TOKENS_MAIN:
            continue
        main_wins.append({'position': (start + end) / 2 / T, 'chunk': chunk})

    # Build secondary windows
    sec_wins = []
    i = 0
    while True:
        start = i * SEC_STEP
        if start >= T:
            break
        end   = min(start + SEC_WINDOW, T)
        chunk = tokens[start:end]
        if not chunk:
            break
        sec_wins.append({'position': (start + end) / 2 / T, 'chunk': chunk})
        i += 1

    # Flatten ALL subchunks from ALL windows into one list
    all_texts  = []
    win_slices = []  # (start_idx, end_idx) into all_texts per window

    for w in main_wins + sec_wins:
        chunk     = w['chunk']
        subchunks = [
            ' '.join(chunk[j:j + SUBCHUNK_WORDS])
            for j in range(0, len(chunk), SUBCHUNK_WORDS)
            if chunk[j:j + SUBCHUNK_WORDS]
        ]
        s = len(all_texts)
        all_texts.extend(subchunks)
        win_slices.append((s, len(all_texts)))

    # Single pipe() call for the entire script
    results = pipe(all_texts, batch_size=batch_size)

    # Distribute scores back to each window
    for w, (s, e) in zip(main_wins + sec_wins, win_slices):
        w['raw_score'] = float(np.mean([_signed(r) for r in results[s:e]]))

    # Z-score main arc
    if main_wins:
        raws = np.array([w['raw_score'] for w in main_wins], dtype=float)
        std  = raws.std()
        zs   = (raws - raws.mean()) / std if std > 0 else raws - raws.mean()
        for w, z in zip(main_wins, zs):
            w['z_score'] = float(z)

    # Savitzky-Golay smooth secondary arc
    if sec_wins:
        raws = np.array([w['raw_score'] for w in sec_wins], dtype=float)
        wl   = min(SG_WINDOW, len(raws))
        if wl % 2 == 0:
            wl -= 1
        smoothed = (
            savgol_filter(raws, wl, min(SG_ORDER, wl - 1)).tolist()
            if wl >= 3 else raws.tolist()
        )
        for w, s in zip(sec_wins, smoothed):
            w['smoothed_score'] = float(s)

    return main_wins, sec_wins


## 3. Load metadata

In [ ]:
import json

corpus = json.loads(METADATA_PATH.read_text())
meta_lookup = {
    e['slug']: {
        'imdb_id': e.get('tmdb', {}).get('imdb_id'),
        'year':    e.get('year'),
        'title':   e.get('title', e['slug']),
    }
    for e in corpus
}
print(f'Loaded metadata for {len(meta_lookup)} scripts')

## 4. Run arc generation

Checkpointing: scripts with an existing `_arc.json` in Drive are skipped. Safe to interrupt and re-run.

In [ ]:
from tqdm.auto import tqdm
import traceback

errors = []

for script_path in tqdm(sorted(script_files), desc='Scripts'):
    slug     = script_path.stem.replace('_parsed', '')
    out_path = ARCS_DIR / f'{slug}_arc.json'

    if out_path.exists():
        continue

    try:
        tokens = tokenize(extract_text(script_path))

        if len(tokens) < 2000:
            errors.append({'slug': slug, 'error': f'Too short ({len(tokens)} tokens)'})
            continue

        main_arc, secondary_arc = compute_arcs(tokens)

        meta   = meta_lookup.get(slug, {})
        record = {
            'slug':          slug,
            'title':         meta.get('title'),
            'imdb_id':       meta.get('imdb_id'),
            'year':          meta.get('year'),
            'token_count':   len(tokens),
            'main_arc':      [
                {'position': w['position'], 'z_score': w['z_score']}
                for w in main_arc
            ],
            'secondary_arc': [
                {'position': w['position'], 'smoothed_score': w['smoothed_score']}
                for w in secondary_arc
            ],
        }
        out_path.write_text(json.dumps(record, indent=2))

    except Exception as e:
        errors.append({'slug': slug, 'error': traceback.format_exc()})
        print(f'ERROR: {slug} — {e}')

print(f'\nDone. {len(errors)} errors.')
if errors:
    (ARCS_DIR / 'errors.json').write_text(json.dumps(errors, indent=2))
    print('Error log -> outputs/arcs/errors.json')


## 5. Consolidate into one file

In [ ]:
all_arcs  = [json.loads(f.read_text()) for f in sorted(ARCS_DIR.glob('*_arc.json'))]
out_all   = ARCS_DIR / 'arcs_all.json'
out_all.write_text(json.dumps(all_arcs, indent=2))

complete = sum(1 for a in all_arcs if len(a['main_arc']) == 20)
partial  = sum(1 for a in all_arcs if 0 < len(a['main_arc']) < 20)
print(f'Consolidated {len(all_arcs)} arcs -> {out_all}')
print(f'Full 20-window: {complete}  |  Partial: {partial}')

## 6. Quick verification — spot-check one script

In [ ]:
import matplotlib.pyplot as plt

sample = all_arcs[0]
print(f"{sample['title']} ({sample['year']}) — {sample['imdb_id']}")
print(f"Tokens: {sample['token_count']}  |  Main windows: {len(sample['main_arc'])}  |  Secondary windows: {len(sample['secondary_arc'])}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3))

axes[0].plot([p['position'] for p in sample['main_arc']],
             [p['z_score']  for p in sample['main_arc']], marker='o', markersize=4)
axes[0].axhline(0, color='gray', lw=1, alpha=0.6)
axes[0].set_title(f"{sample['title']} — Main arc (z-scored)")
axes[0].set_xlabel('Story progress'); axes[0].set_ylabel('Z-score'); axes[0].grid(True, alpha=0.4)

axes[1].plot([p['position']       for p in sample['secondary_arc']],
             [p['smoothed_score'] for p in sample['secondary_arc']], linewidth=1)
axes[1].axhline(0, color='gray', lw=1, alpha=0.6)
axes[1].set_title(f"{sample['title']} — Secondary arc (smoothed)")
axes[1].set_xlabel('Story progress'); axes[1].set_ylabel('Signed sentiment'); axes[1].grid(True, alpha=0.4)

plt.tight_layout()
plt.show()